# NOTEBOOK : 03_player_split_train.ipynb
# INPUT    : ../../data/processed/player_features.csv
# OUTPUT   : ../artifacts/models/player_candidates.joblib

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

ROOT = Path("../..")
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = Path("../artifacts/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / "player_features.csv", low_memory=False)

In [2]:
# 1. Chronological Split
TRAIN_SEASONS = ["2020-21", "2021-22", "2022-23", "2023-24"]
VAL_SEASONS = ["2024-25"]
TEST_SEASONS = ["2025-26"]

train_df = df[df["season"].isin(TRAIN_SEASONS)].copy()
val_df = df[df["season"].isin(VAL_SEASONS)].copy()
test_df = df[df["season"].isin(TEST_SEASONS)].copy()
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

Train: 98,129 | Val: 26,993 | Test: 29,056


In [3]:
# 2. Features & Target
FEATURE_COLS = (
    [c for c in df.columns if c.endswith("_roll3") or c.endswith("_roll5")]
    + ["was_home", "opponent_defensive_strength", "pos_GK", "pos_DEF", "pos_MID", "pos_FWD"]
)
TARGET_COL = "target_points"

X_train, y_train = train_df[FEATURE_COLS].values, train_df[TARGET_COL].values
X_val, y_val = val_df[FEATURE_COLS].values, val_df[TARGET_COL].values

In [4]:
# 3. StandardScaler fitted strictly on Train
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)


def eval_reg(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "mae": round(float(mean_absolute_error(y_true, y_pred)), 4),
        "rmse": round(float(np.sqrt(mse)), 4),
        "mse": round(float(mse), 4),
        "r2": round(float(r2_score(y_true, y_pred)), 4),
    }


models = {}
val_results = []

In [5]:
# (1) Multiple Linear Regression
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
models["linear_regression"] = lr
val_results.append({"model": "linear_regression", **eval_reg(y_val, lr.predict(X_val_sc))})

In [6]:
# (2) Ridge Regression (L2, tune alpha on validation)
best_ridge, best_ridge_rmse, best_ridge_alpha = None, np.inf, None
for alpha in [0.1, 1.0, 10.0, 50.0, 100.0, 500.0]:
    m = Ridge(alpha=alpha, random_state=42)
    m.fit(X_train_sc, y_train)
    rmse = np.sqrt(mean_squared_error(y_val, m.predict(X_val_sc)))
    if rmse < best_ridge_rmse:
        best_ridge_rmse, best_ridge, best_ridge_alpha = rmse, m, alpha
models["ridge"] = best_ridge
val_results.append({"model": f"ridge (alpha={best_ridge_alpha})", **eval_reg(y_val, best_ridge.predict(X_val_sc))})

In [7]:
# (3) Lasso Regression (L1, tune alpha on validation)
best_lasso, best_lasso_rmse, best_lasso_alpha = None, np.inf, None
for alpha in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]:
    m = Lasso(alpha=alpha, max_iter=5000, random_state=42)
    m.fit(X_train_sc, y_train)
    rmse = np.sqrt(mean_squared_error(y_val, m.predict(X_val_sc)))
    if rmse < best_lasso_rmse:
        best_lasso_rmse, best_lasso, best_lasso_alpha = rmse, m, alpha
models["lasso"] = best_lasso
val_results.append({"model": f"lasso (alpha={best_lasso_alpha})", **eval_reg(y_val, best_lasso.predict(X_val_sc))})

In [8]:
# (4) Random Forest Regressor (Bagging)
rf = RandomForestRegressor(
    n_estimators=200, max_depth=10, min_samples_leaf=10, n_jobs=-1, random_state=42
)
rf.fit(X_train_sc, y_train)
models["random_forest"] = rf
val_results.append({"model": "random_forest", **eval_reg(y_val, rf.predict(X_val_sc))})

In [9]:
# (5) XGBoost Regressor (Boosting)
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train_sc, y_train, eval_set=[(X_val_sc, y_val)], verbose=False)
models["xgboost"] = xgb
val_results.append({"model": "xgboost", **eval_reg(y_val, xgb.predict(X_val_sc))})

val_table = pd.DataFrame(val_results).sort_values("rmse")
print("\nValidation Results (2024-25):")
print(val_table.to_string(index=False))

joblib.dump(
    {
        "models": models,
        "scaler": scaler,
        "feature_cols": FEATURE_COLS,
        "best_ridge_alpha": best_ridge_alpha,
        "best_lasso_alpha": best_lasso_alpha,
    },
    MODELS_DIR / "player_candidates.joblib",
)
print("Saved player_candidates.joblib")


Validation Results (2024-25):
              model    mae   rmse    mse     r2
  linear_regression 1.0927 2.0508 4.2056 0.2723
  ridge (alpha=0.1) 1.0927 2.0508 4.2056 0.2723
            xgboost 1.0635 2.0510 4.2065 0.2722
lasso (alpha=0.001) 1.0930 2.0531 4.2152 0.2707
      random_forest 1.0711 2.0790 4.3222 0.2522
Saved player_candidates.joblib
